# **Installing Necessary Libraries**

In [ ]:
!pip install -q -U sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.2 MB/s eta 0:00:00


# **Importing the libraries**

In [ ]:
import os
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline

# **Mounting Google Drive and Defining Project Paths**


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/RAG_DSPy")
DATA_DIR = PROJECT_DIR / "data"
DOCUMENTS_DIR = DATA_DIR / "documents"
RESULTS_DIR = PROJECT_DIR / "results"

print("Project:", PROJECT_DIR)
print("Documents:", DOCUMENTS_DIR)
print("Evaluation file:", DATA_DIR / "evaluation_questions.csv")

print("\nDocuments found:")
for file in DOCUMENTS_DIR.iterdir():
    print("-", file.name)

Project: /content/drive/MyDrive/RAG_DSPy
Documents: /content/drive/MyDrive/RAG_DSPy/data/documents
Evaluation file: /content/drive/MyDrive/RAG_DSPy/data/evaluation_questions.csv

Documents found:
- rag_vector_databases.txt
- rag_indexes.txt
- rag_fundamentals.txt


# **Loading the documents**

This cell reads all .txt files from the documents directory and stores each document together with its filename.

The filename is retained as source metadata so that retrieved information can later be traced back to its original document.

In [ ]:
documents = []

for file in DOCUMENTS_DIR.glob("*.txt"):
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "source": file.name,
        "text": text
    })

print(f"Number of documents loaded: {len(documents)}")

for doc in documents:
    print(f"- {doc['source']}: {len(doc['text'])} characters")

Number of documents loaded: 3
- rag_vector_databases.txt: 2752 characters
- rag_indexes.txt: 2414 characters
- rag_fundamentals.txt: 2960 characters


# **Creating Document Chunks**



The documents are split into sentences and grouped into chunks of approximately five sentences. One sentence is shared between consecutive chunks to provide overlap and preserve context across chunk boundaries.


In [ ]:
import re

chunks = []

chunk_size = 5
chunk_overlap = 1

for doc in documents:
    sentences = re.split(r'(?<=[.!?])\s+', doc["text"].strip())

    start = 0

    while start < len(sentences):
        end = min(start + chunk_size, len(sentences))

        chunk_text = " ".join(sentences[start:end])

        chunks.append({
            "source": doc["source"],
            "text": chunk_text
        })

        start += chunk_size - chunk_overlap

print(f"Total chunks created: {len(chunks)}")

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i} ---")
    print(f"Source: {chunk['source']}")
    print(f"Text: {chunk['text']}")

Total chunks created: 29

--- Chunk 0 ---
Source: rag_vector_databases.txt
Text: RAG, EMBEDDINGS, AND VECTOR DATABASES

Source: Microsoft Learn, Generative AI for Beginners — Retrieval Augmented Generation (RAG) and Vector Databases. Purpose: Learning material for a RAG demonstration project. 1. Why use RAG? A language model has knowledge learned during training, but an application may need to answer questions using a particular collection of documents.

--- Chunk 1 ---
Source: rag_vector_databases.txt
Text: A language model has knowledge learned during training, but an application may need to answer questions using a particular collection of documents. RAG provides a way to connect an LLM with external information. Instead of asking the LLM to answer only from its internal learned knowledge, the application first searches the application's data and supplies relevant results to the model. 2. Vector embeddings

An embedding converts text into a numerical vector.

--- Chunk 2 ---
Source:

# **Loading the Embedding Model**

This cell loads all-MiniLM-L6-v2 which is a Sentence Transformer model used to convert text into numerical vector representations called embeddings to capture semantic meaning, allowing similar questions and document chunks to be compared mathematically.

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


# **Creating embeddings for the Document Chunks**

In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Number of embeddings:", len(chunk_embeddings))
print("Embedding dimensions:", chunk_embeddings.shape[1])

Number of embeddings: 29
Embedding dimensions: 384


# **Creating the FAISS vector index**

This cell creates a FAISS vector index and stores the document embeddings inside it.

FAISS will later use this index to retrieve the chunks that are most semantically similar to a user's question.

In [ ]:
embedding_dim = chunk_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(chunk_embeddings.astype("float32"))

print("FAISS index created successfully!")
print("Number of vectors in index:", index.ntotal)

FAISS index created successfully!
Number of vectors in index: 29


# **Testing FAISS Retrieval**

This cell tests thether the vector database can retrieve information relevant to the question.

The question is converted into an embedding and compared against all document embeddings.

FAISS returns the top 3 chunks with the highest similarity scores.



In [ ]:
query = "What is Retrieval-Augmented Generation?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

k = 3

scores, indices = index.search(
    query_embedding.astype("float32"),
    k
)

print(f"Query: {query}\n")

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"--- Result {rank} ---")
    print(f"Similarity score: {score:.4f}")
    print(f"Source: {chunks[idx]['source']}")
    print(f"Text: {chunks[idx]['text']}")
    print()

Query: What is Retrieval-Augmented Generation?

--- Result 1 ---
Similarity score: 0.6240
Source: rag_fundamentals.txt
Text: Retrieval-Augmented Generation (RAG) is an architecture that combines information retrieval with a large language model (LLM). Instead of relying only on knowledge learned during model training, the application retrieves relevant information from an external knowledge source and provides that information to the LLM as context. The basic idea is:

User question
-> retrieve relevant information
-> add the retrieved information to the prompt
-> generate an answer using the LLM

RAG is useful when answers need to be based on private documents, domain-specific information, or information that can change after the model was trained. 2. The RAG workflow

A traditional RAG workflow has three main stages:

Retrieve:
The application receives a user question and searches an index or data store for relevant information.

--- Result 2 ---
Similarity score: 0.6217
Source: rag_

# **Loading Qwen**

This cell loads Qwen/Qwen2.5-3B-Instruct, which is used as the generative component of the RAG pipeline.

The model receives the retrieved document context together with the user's question and generates the final answer.

In [ ]:
model_id = "Qwen/Qwen2.5-3B-Instruct"

generator = pipeline(
    "text-generation",
    model=model_id,
    device_map="auto"
)

print("Qwen loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Qwen loaded successfully!


# **Loading the Qwen Tokenizer**

The tokenizer converts text into the token representation expected by the Qwen language model.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct"
)

print("Qwen tokenizer loaded successfully!")

Qwen tokenizer loaded successfully!


# **Retrieving Context and Generating RAG Answer**

This cell demonstrates the complete basic RAG pipeline for one question.

The process is:

1. Convert the question into an embedding.
2. Retrieve the top 3 relevant document chunks using FAISS.
3. Combine the retrieved chunks into a context.
4. Provide the context and question to Qwen.
5. Generate an answer using only the retrieved context.

This connects the retrieval and generation components into an end-to-end RAG system.

In [ ]:
query = "What is Retrieval-Augmented Generation?"

# 1. Convert question into an embedding
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 2. Retrieve top 3 relevant chunks
k = 3

scores, indices = index.search(
    query_embedding.astype("float32"),
    k
)

# 3. Get retrieved text
retrieved_chunks = [chunks[idx]["text"] for idx in indices[0]]

# 4. Combine retrieved chunks into context
context = "\n\n".join(retrieved_chunks)

# 5. Create a chat-formatted message for Qwen
messages = [
    {
        "role": "system",
        "content": "Answer the user's question using only the provided context. If the context does not contain enough information, say that the information is not available in the context."
    },
    {
        "role": "user",
        "content": f"""Context:

{context}

Question:
{query}"""
    }
]

# 6. Generate the answer
result = generator(
    messages,
    max_new_tokens=150,
    do_sample=False
)

# 7. Extract the response
rag_answer = result[0]["generated_text"][-1]["content"].strip()

print("RAG Answer:")
print(rag_answer)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG Answer:
Retrieval-Augmented Generation (RAG) is an architecture that combines information retrieval with a large language model (LLM).


# **Creating the RAG Function**

This cell converts the previous end-to-end RAG process into a reusable Python function

In [ ]:
def generate_rag_answer(query, k=3):

    # 1. Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # 2. Retrieve the top k relevant chunks
    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    # 3. Get the retrieved chunk text
    retrieved_chunks = [chunks[idx]["text"] for idx in indices[0]]

    # 4. Combine chunks into context
    context = "\n\n".join(retrieved_chunks)

    # 5. Create chat messages for Qwen
    messages = [
        {
            "role": "system",
            "content": (
                "Answer the user's question using only the provided context. "
                "If the context does not contain enough information, "
                "say that the information is not available in the context."
            )
        },
        {
            "role": "user",
            "content": f"""Context:

{context}

Question:
{query}"""
        }
    ]

    # 6. Generate answer
    result = generator(
        messages,
        max_new_tokens=150,
        do_sample=False
    )

    # 7. Extract the assistant's response
    answer = result[0]["generated_text"][-1]["content"].strip()

    return answer

In [ ]:
answer = generate_rag_answer("What is RAG?")
print(answer)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG is an architecture that combines information retrieval with a large language model (LLM).


# **Testing RAG on the Evaluation Set**

In [ ]:
import pandas as pd

eval_path = "/content/drive/MyDrive/RAG_DSPy/data/evaluation_questions.csv"

eval_df = pd.read_csv(eval_path)

rag_results = []

for _, row in eval_df.iterrows():

    question = row["question"]

    answer = generate_rag_answer(question)

    rag_results.append({
        "question_id": row["question_id"],
        "question": question,
        "reference_answer": row["reference_answer"],
        "generated_answer": answer
    })

    print(f"Completed {row['question_id']}")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed q01


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed q02


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed q03


... (38 outputs left out)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed q23


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed q24
Completed q25


# **Converting RAG results to a DataFrame**

In [ ]:
rag_df = pd.DataFrame(rag_results)

print(f"RAG results generated: {len(rag_df)}")

display(rag_df.head(5))

RAG results generated: 25


,question_id,question,reference_answer,generated_answer
0,q01,What is Retrieval-Augmented Generation (RAG)?,RAG is an architecture that combines informati...,Retrieval-Augmented Generation (RAG) is an arc...
1,q02,Why is RAG useful for an LLM application?,RAG allows an application to provide an LLM wi...,RAG is useful for an LLM application because t...
2,q03,What are the three main stages of a basic RAG ...,"The three main stages are retrieve, augment, a...",The three main stages of a basic RAG workflow ...
3,q04,What happens during the retrieval stage of RAG?,The system searches an index or data store for...,"During the retrieve stage of RAG, an index or ..."
4,q05,What happens during the augmentation stage of ...,The system combines the user's question with t...,"During the augmentation stage of RAG, the user..."


# **Saving RAG Results**

In [ ]:
rag_results_path = RESULTS_DIR / "rag_results.csv"

rag_df.to_csv(rag_results_path, index=False)

print(f"Saved RAG results to: {rag_results_path}")

Saved RAG results to: /content/drive/MyDrive/RAG_DSPy/results/rag_results.csv
